In [1]:
# 01_data_check.ipynb

In [2]:
# Cell 1: imports and display setup

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 60)

DATA_PATH = Path("../data/raw/final_crop_dataset.csv")
UNIT_COL = "adm_id"
TIME_COL = "harvest_year"
TARGET_COL = "yield"

In [3]:
# Cell 2: load data and normalize column names

if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)
else:
    raise ValueError(f"Unsupported file type: {DATA_PATH.suffix}")

# normalize headers
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (6715, 28)
Columns: ['crop_name', 'country_code', 'adm_id', 'harvest_year', 'yield', 'production', 'harvest_area', 'sos', 'eos', 'awc', 'bulk_density', 'drainage_class', 'latitude', 'longitude', 'region_area', 'crop_area', 'crop_area_percentage', 'avg_tmin', 'avg_tmax', 'avg_tavg', 'avg_rad', 'avg_et0', 'avg_vpd', 'avg_cwb', 'avg_ssm', 'avg_rsm', 'avg_ndvi', 'avg_fpar']


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,awc,bulk_density,drainage_class,latitude,longitude,region_area,crop_area,crop_area_percentage,avg_tmin,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
0,wheat,IN,IN-14-0001,2003,0.770,15050,19550,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.711417,30.860150,23.710323,1.913480e+07,4.349425,32.732055,-3.986236,3.784343,254.632300,0.330688,27.080077
1,wheat,IN,IN-14-0001,2004,0.605,11590,19160,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,15.942244,29.781677,22.634142,1.934158e+07,4.233874,30.660197,-3.651843,5.570850,292.739850,0.408625,35.197833
2,wheat,IN,IN-14-0001,2005,0.605,11590,19160,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.505570,30.305594,23.347766,1.931177e+07,4.289164,31.543805,-3.535000,4.622133,265.181766,0.378812,31.107846
3,wheat,IN,IN-14-0001,2006,0.754,14400,19090,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,15.556709,30.182913,22.754858,1.971577e+07,4.289583,31.487173,-3.834213,4.938567,269.223709,0.373188,31.081538
4,wheat,IN,IN-14-0001,2007,0.963,19430,20170,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.346071,30.354496,23.244661,1.954419e+07,4.294756,31.479134,-4.076094,5.065094,272.097992,0.351313,29.904000


In [4]:
# Cell 3: quick schema and duplicate-column check

print("Columns:")
display(pd.Series(df.columns, name="column_name"))

print("\nDtypes:")
display(df.dtypes.rename("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate column names:", pd.Series(df.columns).duplicated().sum())

Columns:


0                crop_name
1             country_code
2                   adm_id
3             harvest_year
4                    yield
5               production
6             harvest_area
7                      sos
8                      eos
9                      awc
10            bulk_density
11          drainage_class
12                latitude
13               longitude
14             region_area
15               crop_area
16    crop_area_percentage
17                avg_tmin
18                avg_tmax
19                avg_tavg
20                 avg_rad
21                 avg_et0
22                 avg_vpd
23                 avg_cwb
24                 avg_ssm
25                 avg_rsm
26                avg_ndvi
27                avg_fpar
Name: column_name, dtype: str


Dtypes:


crop_name                   str
country_code                str
adm_id                      str
harvest_year              int64
yield                   float64
production                int64
harvest_area              int64
sos                     float64
eos                     float64
awc                     float64
bulk_density            float64
drainage_class            int64
latitude                float64
longitude               float64
region_area               int64
crop_area               float64
crop_area_percentage    float64
avg_tmin                float64
avg_tmax                float64
avg_tavg                float64
avg_rad                 float64
avg_et0                 float64
avg_vpd                 float64
avg_cwb                 float64
avg_ssm                 float64
avg_rsm                 float64
avg_ndvi                float64
avg_fpar                float64
Name: dtype, dtype: object


Duplicate rows: 0
Duplicate column names: 0


In [5]:
# Cell 4: missingness and basic numeric summary

missing_tbl = (
    df.isna()
      .sum()
      .rename("n_missing")
      .to_frame()
      .assign(pct_missing=lambda x: 100 * x["n_missing"] / len(df))
      .sort_values(["pct_missing", "n_missing"], ascending=False)
)

display(missing_tbl.head(30))

num_summary = df.describe(include=[np.number]).T
display(num_summary)

,n_missing,pct_missing
crop_name,0,0.0
country_code,0,0.0
adm_id,0,0.0
harvest_year,0,0.0
yield,0,0.0
production,0,0.0
harvest_area,0,0.0
sos,0,0.0
eos,0,0.0
awc,0,0.0


,count,mean,std,min,25%,50%,75%,max
harvest_year,6715.0,2.010115e+03,4.308095e+00,2.003000e+03,2.006000e+03,2.010000e+03,2.014000e+03,2.017000e+03
yield,6715.0,2.305038e+00,1.133131e+00,1.070000e-01,1.379500e+00,2.100000e+00,3.078500e+00,5.860000e+00
production,6715.0,1.972937e+05,2.654897e+05,0.000000e+00,6.540000e+03,8.690000e+04,2.917700e+05,1.969000e+06
harvest_area,6715.0,6.466934e+04,6.937080e+04,0.000000e+00,4.550000e+03,4.292000e+04,9.866500e+04,3.970000e+05
sos,6715.0,3.134138e+02,9.037540e+01,4.570000e-01,3.354140e+02,3.401880e+02,3.438220e+02,3.644820e+02
eos,6715.0,1.088654e+02,1.653997e+01,8.227100e+01,1.009650e+02,1.043710e+02,1.107380e+02,1.868180e+02
awc,6715.0,1.367049e+01,1.771508e+00,9.043000e+00,1.256200e+01,1.387500e+01,1.467600e+01,2.280800e+01
bulk_density,6715.0,1.498376e+00,7.127975e-02,1.076000e+00,1.451000e+00,1.504000e+00,1.551000e+00,1.661000e+00
drainage_class,6715.0,4.592554e+00,6.848751e-01,2.000000e+00,4.000000e+00,5.000000e+00,5.000000e+00,6.000000e+00
latitude,6715.0,2.464672e+01,3.822398e+00,1.004000e+01,2.196630e+01,2.507111e+01,2.707407e+01,3.257659e+01


In [6]:
# Cell 5: panel sanity checks for time coverage and unit consistency

required_cols = [UNIT_COL, TIME_COL, TARGET_COL, "crop_name"]
missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}\n"
        f"Available columns are: {df.columns.tolist()}"
    )

df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce")

panel_check = (
    df.groupby(UNIT_COL)
      .agg(
          n_years=(TIME_COL, "nunique"),
          min_year=(TIME_COL, "min"),
          max_year=(TIME_COL, "max"),
          n_rows=(TIME_COL, "size"),
          cropname_nunique=("crop_name", "nunique"),
          yield_missing=(TARGET_COL, lambda s: s.isna().sum())
      )
      .sort_values(["n_years", "n_rows", "yield_missing"], ascending=[True, True, False])
)

display(panel_check.head(20))
display(panel_check.describe())

,n_years,min_year,max_year,n_rows,cropname_nunique,yield_missing
adm_id,,,,,,
IN-01-0047,1,2017,2017,1,1,0
IN-01-0049,1,2010,2010,1,1,0
IN-01-0503,1,2017,2017,1,1,0
IN-05-0065,1,2008,2008,1,1,0
IN-05-0067,1,2008,2008,1,1,0
IN-05-0068,1,2008,2008,1,1,0
IN-05-0069,1,2008,2008,1,1,0
IN-05-0080,1,2008,2008,1,1,0
IN-05-0082,1,2008,2008,1,1,0


,n_years,min_year,max_year,n_rows,cropname_nunique,yield_missing
count,501.000000,501.000000,501.000000,501.000000,501.0,501.0
mean,13.403194,2004.171657,2016.670659,13.403194,1.0,0.0
std,3.775329,3.185981,1.430146,3.775329,0.0,0.0
min,1.000000,2003.000000,2008.000000,1.000000,1.0,0.0
25%,15.000000,2003.000000,2017.000000,15.000000,1.0,0.0
50%,15.000000,2003.000000,2017.000000,15.000000,1.0,0.0
75%,15.000000,2003.000000,2017.000000,15.000000,1.0,0.0
max,15.000000,2017.000000,2017.000000,15.000000,1.0,0.0
